# DenseNet121 Chest X-Ray Classification — Kaggle Training Notebook

This notebook trains a binary DenseNet121 transfer-learning model on a licensed folder-based chest-X-ray
dataset with `NORMAL` and `PNEUMONIA` class folders. It includes data audit, class-imbalance handling,
training, evaluation, threshold analysis, error analysis, optional Grad-CAM, and deployment exports.

> **Medical disclaimer:** This notebook is for educational and portfolio demonstration only. It is not a
> medical diagnostic tool. Do not upload or publish private, identifiable, or confidential medical images.
> Any real-world medical use requires clinical validation and qualified professional review.

## 1. Configuration

Attach a dataset in Kaggle with this structure:

```text
chest_xray/train/NORMAL
chest_xray/train/PNEUMONIA
chest_xray/val/NORMAL       # or validation
chest_xray/val/PNEUMONIA
chest_xray/test/NORMAL
chest_xray/test/PNEUMONIA
```

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
FROZEN_EPOCHS = 8
FINE_TUNE_EPOCHS = 3
THRESHOLD = 0.50
OUTPUT_DIR = Path('/kaggle/working/densenet_medical_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 2. Locate the attached dataset

The next cell searches common Kaggle input locations. Set `DATASET_ROOT` manually when needed.

In [ ]:
def find_dataset_root(base=Path('/kaggle/input')):
    candidates = []
    for train_dir in base.rglob('train'):
        if (train_dir / 'NORMAL').exists() and (train_dir / 'PNEUMONIA').exists():
            root = train_dir.parent
            if (root / 'test').exists():
                candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            'No compatible dataset found. Attach a dataset and set DATASET_ROOT manually.'
        )
    return sorted(candidates, key=lambda path: len(str(path)))[0]

DATASET_ROOT = find_dataset_root()
TRAIN_DIR = DATASET_ROOT / 'train'
VAL_DIR = DATASET_ROOT / ('val' if (DATASET_ROOT / 'val').exists() else 'validation')
TEST_DIR = DATASET_ROOT / 'test'
print('Dataset root:', DATASET_ROOT)
print('Train:', TRAIN_DIR)
print('Validation:', VAL_DIR)
print('Test:', TEST_DIR)

## 3. Dataset audit

This checks class counts, readable images, common dimensions, and exact duplicate files. Review duplicate
groups carefully; patient-level leakage requires patient identifiers, which may not be available from filenames.

In [ ]:
SUPPORTED = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}

def audit_split(split_dir: Path):
    counts = Counter()
    corrupt = []
    dimensions = Counter()
    hashes = defaultdict(list)
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        for path in class_dir.rglob('*'):
            if not path.is_file() or path.suffix.lower() not in SUPPORTED:
                continue
            try:
                with Image.open(path) as image:
                    image.verify()
                with Image.open(path) as image:
                    dimensions[image.size] += 1
                hashes[hashlib.sha256(path.read_bytes()).hexdigest()].append(str(path))
                counts[class_dir.name] += 1
            except (UnidentifiedImageError, OSError, ValueError):
                corrupt.append(str(path))
    duplicate_groups = [paths for paths in hashes.values() if len(paths) > 1]
    return counts, corrupt, dimensions, duplicate_groups

audits = {}
for name, directory in {'train': TRAIN_DIR, 'validation': VAL_DIR, 'test': TEST_DIR}.items():
    counts, corrupt, dimensions, duplicates = audit_split(directory)
    audits[name] = {
        'counts': dict(counts),
        'corrupt_count': len(corrupt),
        'duplicate_group_count': len(duplicates),
        'common_dimensions': [(str(size), count) for size, count in dimensions.most_common(5)],
    }
print(json.dumps(audits, indent=2))

In [ ]:
class_rows = []
for split, details in audits.items():
    for class_name, count in details['counts'].items():
        class_rows.append({'split': split, 'class': class_name, 'count': count})
class_distribution = pd.DataFrame(class_rows)
display(class_distribution)

pivot = class_distribution.pivot(index='split', columns='class', values='count').fillna(0)
pivot.plot(kind='bar', figsize=(9, 5), title='Class Distribution by Split')
plt.ylabel('Images')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=160)
plt.show()

## 4. Preview images

Only public, de-identified images allowed by the dataset license should appear in a published notebook.

In [ ]:
def sample_paths(directory: Path, class_name: str, count=4):
    paths = [path for path in (directory / class_name).rglob('*') if path.suffix.lower() in SUPPORTED]
    return random.sample(paths, min(count, len(paths)))

classes = sorted(path.name for path in TRAIN_DIR.iterdir() if path.is_dir())
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3 * len(classes)))
axes = np.atleast_2d(axes)
for row, class_name in enumerate(classes):
    for col, path in enumerate(sample_paths(TRAIN_DIR, class_name, 4)):
        with Image.open(path) as image:
            axes[row, col].imshow(image, cmap='gray')
        axes[row, col].set_title(class_name)
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_input_images.png', dpi=160)
plt.show()

## 5. TensorFlow datasets and class weights

The folder loader infers class order alphabetically. Save the exact order with the model metadata.
Class weights are calculated from the training set to reduce majority-class dominance.

In [ ]:
def make_dataset(directory: Path, shuffle: bool):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='binary',
        color_mode='rgb',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

train_ds = make_dataset(TRAIN_DIR, shuffle=True)
val_ds = make_dataset(VAL_DIR, shuffle=False)
test_ds = make_dataset(TEST_DIR, shuffle=False)
CLASS_NAMES = train_ds.class_names
print('Class order:', CLASS_NAMES)

train_counts = audits['train']['counts']
y_for_weights = []
for index, name in enumerate(CLASS_NAMES):
    y_for_weights.extend([index] * int(train_counts[name]))
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=np.asarray(y_for_weights))
CLASS_WEIGHT = {index: float(value) for index, value in enumerate(weights)}
print('Class weights:', CLASS_WEIGHT)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 6. Conservative augmentation and DenseNet121 model

Horizontal flips are excluded by default because laterality may matter. Rotations, shifts, zoom, and contrast
changes are deliberately small to avoid unrealistic anatomy.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.025),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomTranslation(0.02, 0.02),
    tf.keras.layers.RandomContrast(0.08),
], name='conservative_augmentation')

inputs = tf.keras.Input((*IMAGE_SIZE, 3), name='image')
x = augmentation(inputs)
x = tf.keras.applications.densenet.preprocess_input(x)
backbone = tf.keras.applications.DenseNet121(
    include_top=False,
    weights='imagenet',
    input_shape=(*IMAGE_SIZE, 3),
)
backbone.trainable = False
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = tf.keras.layers.Dropout(0.35, name='dropout')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='pneumonia_probability')(x)
model = tf.keras.Model(inputs, outputs, name='densenet121_chest_xray_classifier')
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='roc_auc'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)
model.summary()

## 7. Frozen-backbone training

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        OUTPUT_DIR / 'best_frozen_model.keras', monitor='val_roc_auc', mode='max', save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_roc_auc', mode='max', patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7
    ),
]

history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FROZEN_EPOCHS,
    callbacks=callbacks,
    class_weight=CLASS_WEIGHT,
)

## 8. Optional fine-tuning

Only the final DenseNet layers are unfrozen. Batch-normalization layers remain frozen, and the learning rate
is reduced substantially.

In [ ]:
backbone.trainable = True
for layer in backbone.layers[:-40]:
    layer.trainable = False
for layer in backbone.layers[-40:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='roc_auc'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks,
    class_weight=CLASS_WEIGHT,
)

## 9. Training curves

In [ ]:
def merge_histories(*histories):
    keys = set().union(*(history.history.keys() for history in histories))
    return {key: sum((history.history.get(key, []) for history in histories), []) for key in keys}

history = merge_histories(history_frozen, history_fine)
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)

for metric in ['accuracy', 'loss', 'roc_auc', 'pr_auc']:
    if metric not in history:
        continue
    plt.figure(figsize=(8, 5))
    plt.plot(history[metric], label=f'train_{metric}')
    if f'val_{metric}' in history:
        plt.plot(history[f'val_{metric}'], label=f'val_{metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'training_{metric}.png', dpi=160)
    plt.show()

## 10. Test predictions and comprehensive metrics

In [ ]:
y_true = []
y_score = []
image_batches = []
for images, labels in test_ds:
    scores = model.predict(images, verbose=0).reshape(-1)
    y_true.extend(labels.numpy().reshape(-1).astype(int).tolist())
    y_score.extend(scores.tolist())
    if len(image_batches) < 5:
        image_batches.append(images.numpy())
y_true = np.asarray(y_true)
y_score = np.asarray(y_score)
y_pred = (y_score >= THRESHOLD).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='binary', zero_division=0
)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'macro_f1': float(report['macro avg']['f1-score']),
    'weighted_f1': float(report['weighted avg']['f1-score']),
    'roc_auc': float(roc_auc_score(y_true, y_score)),
    'pr_auc': float(average_precision_score(y_true, y_score)),
    'threshold': THRESHOLD,
    'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
}
print(json.dumps(metrics, indent=2))
pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / 'classification_report.csv')
with (OUTPUT_DIR / 'model_metrics.json').open('w') as file:
    json.dump(metrics, file, indent=2)

## 11. Confusion matrix, ROC, and precision-recall curves

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=30)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_score)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC={metrics['roc_auc']:.4f}")
plt.plot([0, 1], [0, 1], '--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=160)
plt.show()

pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_score)
plt.figure(figsize=(6, 6))
plt.plot(pr_recall, pr_precision, label=f"PR-AUC={metrics['pr_auc']:.4f}")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'precision_recall_curve.png', dpi=160)
plt.show()

## 12. Threshold analysis

Select thresholds based on the intended research objective and validation data. Never tune on the test set
for a final reported model; this cell is educational and should be moved to validation data in a formal study.

In [ ]:
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    pred = (y_score >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, pred, average='binary', zero_division=0
    )
    threshold_rows.append({'threshold': threshold, 'precision': p, 'recall': r, 'f1': f})
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(OUTPUT_DIR / 'threshold_analysis.csv', index=False)
display(threshold_df.sort_values('f1', ascending=False).head(10))

## 13. Error analysis

Review false positives, false negatives, low-confidence predictions, and high-confidence errors. Avoid
sharing restricted images when publishing the notebook.

In [ ]:
error_df = pd.DataFrame({
    'index': np.arange(len(y_true)),
    'true_label': y_true,
    'predicted_label': y_pred,
    'positive_probability': y_score,
    'confidence': np.maximum(y_score, 1 - y_score),
})
error_df['error_type'] = np.select(
    [
        (error_df.true_label == 0) & (error_df.predicted_label == 1),
        (error_df.true_label == 1) & (error_df.predicted_label == 0),
    ],
    ['false_positive', 'false_negative'],
    default='correct',
)
error_df.to_csv(OUTPUT_DIR / 'prediction_error_analysis.csv', index=False)
display(error_df[error_df.error_type != 'correct'].sort_values('confidence', ascending=False).head(20))

## 14. Optional Grad-CAM

Grad-CAM is a qualitative model-debugging aid, not proof of medical reasoning or correctness.

In [ ]:
def make_gradcam(image_batch, model, class_index=0):
    feature_layer = model.get_layer('densenet121')
    grad_model = tf.keras.Model(model.inputs, [feature_layer.output, model.output])
    with tf.GradientTape() as tape:
        features, prediction = grad_model(image_batch, training=False)
        score = prediction[:, 0]
    gradients = tape.gradient(score, features)
    pooled = tf.reduce_mean(gradients, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(features[0] * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

example_images = image_batches[0][:4]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, image in enumerate(example_images):
    heatmap = make_gradcam(image[None, ...], model)
    axes[0, i].imshow(np.clip(image / 255.0, 0, 1))
    axes[0, i].axis('off')
    axes[1, i].imshow(np.clip(image / 255.0, 0, 1))
    axes[1, i].imshow(heatmap, alpha=0.45, cmap='jet')
    axes[1, i].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam_examples.png', dpi=160)
plt.show()

## 15. Export model and metadata

Move these artifacts into the GitHub project and Hugging Face Space only after verifying the class order,
preprocessing, dataset license, and documentation.

In [ ]:
MODEL_PATH = OUTPUT_DIR / 'densenet_medical_classification_model.keras'
model.save(MODEL_PATH)

model_metadata = {
    'project_name': 'DenseNet Medical Image Classification',
    'artifact_filename': MODEL_PATH.name,
    'architecture': 'DenseNet121 transfer learning',
    'dataset_status': 'public_chest_xray_dataset_user_must_document_source_and_license',
    'dataset_root': str(DATASET_ROOT),
    'classes': CLASS_NAMES,
    'class_to_index': {name: index for index, name in enumerate(CLASS_NAMES)},
    'input_shape': [*IMAGE_SIZE, 3],
    'color_mode': 'RGB',
    'external_preprocessing': 'Resize to 224x224 RGB. DenseNet preprocess_input is embedded in the model graph.',
    'threshold': THRESHOLD,
    'training_configuration': {
        'seed': SEED,
        'batch_size': BATCH_SIZE,
        'frozen_epochs': FROZEN_EPOCHS,
        'fine_tune_epochs': FINE_TUNE_EPOCHS,
        'class_weight': CLASS_WEIGHT,
    },
    'metrics': metrics,
    'medical_disclaimer': (
        'Educational and portfolio demonstration only. Not a diagnostic tool. '
        'Requires clinical validation and qualified professional review.'
    ),
}
with (OUTPUT_DIR / 'model_metadata.json').open('w') as file:
    json.dump(model_metadata, file, indent=2)

print('Exported files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name, path.stat().st_size)

## 16. Final interpretation and limitations

Report the exact dataset source, license, patient-level split method, class counts, and evaluation protocol.
Discuss false negatives and false positives separately. External validation, calibration, subgroup analysis,
robustness checks, and clinical review are required before any healthcare use. This portfolio notebook does
not establish medical safety, efficacy, or generalizability.